Agent with intrupt which gives access to **HUMAN IN LOOP** Objective 



**AI SQL query generator**

Descriptions : An AI agent that converts natural language into SQL queries, but the query is executed only after human approval.

Components:\
**LLM for thinking**\
**System memmory with predefined table details**\
**Use excution function to simulate databse intercation**


*Steps*\
*-Build LLM*\
*-Add tools*(**generate_query(Userquery) , excecute_query(SQLquery)**)\
*-Agent using langgaraph*\
*-Add a interrutp before tool excecution*(**Show the query to the user**)\
*-Once query is approved Run the query*

In [3]:
#Import all the required libaries 
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph , START , END , MessagesState
from langchain.messages import SystemMessage , HumanMessage , AIMessage
from dotenv import load_dotenv

load_dotenv()


True

Creating LLM 

In [2]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview"
)

Create a state 

In [ ]:
class State(MessagesState):
    user_query : str
    query : str

Tools for creating query and writing the query 


In [ ]:
#Create query 

def create_query(state : State) -> State :
    
    system_message =  SystemMessage("""
        You are an AI SQL Query Generator.

        You are provided with the following MySQL database schema:

        DATABASE: RailwayReservationSystem

        TABLE: users
        - user_id (INT, PRIMARY KEY, AUTO_INCREMENT)
        - name (VARCHAR(100))
        - email (VARCHAR(100))
        - phone (VARCHAR(15))
        - password (VARCHAR(255))

        TABLE: trains
        - train_id (INT, PRIMARY KEY, AUTO_INCREMENT)
        - train_name (VARCHAR(100))
        - source_station (VARCHAR(100))
        - destination_station (VARCHAR(100))
        - total_seats (INT)
        - available_seats (INT)

        TABLE: bookings
        - booking_id (INT, PRIMARY KEY, AUTO_INCREMENT)
        - user_id (INT, FOREIGN KEY REFERENCES users(user_id))
        - train_id (INT, FOREIGN KEY REFERENCES trains(train_id))
        - booking_date (DATE)
        - travel_date (DATE)
        - seats_booked (INT)
        - status (VARCHAR(50))

        TABLE: payments
        - payment_id (INT, PRIMARY KEY, AUTO_INCREMENT)
        - booking_id (INT, FOREIGN KEY REFERENCES bookings(booking_id))
        - amount (DECIMAL(10,2))
        - payment_status (VARCHAR(50))
        - payment_date (DATE)

        Rules:
        1. Always understand the user's question carefully.
        2. Generate only valid MySQL queries.
        3. Do not explain the query unless asked.
        4. Use proper JOINs wherever required.
        5. If multiple queries are needed, provide them separately.
        6. Never generate destructive queries like DROP DATABASE or DELETE without explicit request.
        7. Output should contain only the MySQL query.

        You must give the MySQL query for the below question.
        """)


    prompt =  [system_message , HumanMessage(content=state['user_query'])]


    response = llm.invoke(prompt)

    return {
        "query": response.content
    }

# Excecute database operation

def excute_query(state : State):

    print(state['query'])
    return state
